# LADAS2TEI 3 : convertir et vérifier

**Objectif** : produire le fichier TEI, puis s'assurer qu'il est correct.

Ce carnet s'utilise cellule par cellule : cliquez sur une cellule de code, puis faites `Maj + Entrée` pour l'exécuter. Une ligne qui commence par `!` est envoyée au **terminal** : c'est exactement ce que vous taperiez en ligne de commande.

> ⚠️ Le carnet doit être lancé avec le **même environnement virtuel** que celui où vous avez installé les dépendances (partie 01).

## 0. Se placer dans le bon dossier

`python -m ladas2tei.main` ne fonctionne que **depuis le dossier `LADAS2TEI`**. Dans un carnet, on change de dossier avec `%cd` (et non `!cd`, qui n'aurait d'effet que le temps d'une ligne).

Adaptez le chemin si vous avez cloné le dépôt ailleurs.

In [ ]:
%cd ~/LADAS2TEI
!pwd
!ls

Vous devez voir `corpus`, `metadata.csv`, `ladas2tei`, `requirements.txt`… Sinon, revenez à la partie 02.

Pour être sûr d'utiliser le Python de l'environnement virtuel, on récupère son chemin :

In [ ]:
import sys
print(sys.executable)

Ce chemin doit contenir `venv`. Dans les commandes ci-dessous, `{sys.executable}` sera remplacé par ce chemin : c'est l'équivalent de `python` après `source venv/bin/activate`.

## 1. La commande

```
python -m ladas2tei.main corpus -o TEI --metadata-csv metadata.csv
```

| Morceau | Rôle |
|---|---|
| `python -m ladas2tei.main` | lance le programme (module `main` du paquet `ladas2tei`) |
| `corpus` | dossier **parent** contenant un sous-dossier par document |
| `-o TEI` | dossier de sortie |
| `--metadata-csv metadata.csv` | fichier de métadonnées (obligatoire dans ce mode) |

Commençons par lire l'aide du programme :

In [ ]:
!{sys.executable} -m ladas2tei.main --help

Puis lançons la conversion :

In [ ]:
!{sys.executable} -m ladas2tei.main corpus -o TEI --metadata-csv metadata.csv

Résultat attendu : un fichier `faux_visage_ONB_10790D2D.xml` dans le dossier `TEI`.

In [ ]:
!ls -lh TEI

## 2. Convertir un seul document

On peut aussi viser directement le dossier ALTO, en donnant un titre :

In [ ]:
!{sys.executable} -m ladas2tei.main corpus/faux_visage_ONB_10790D2D/alto -o faux_visage.xml --title "Faux visage"

On peut même désigner quelques pages seulement. Listez d'abord les fichiers pour choisir :

In [ ]:
!ls corpus/faux_visage_ONB_10790D2D/alto | head -n 5

Puis remplacez `<page_1>` et `<page_2>` par deux noms de fichiers de la liste :

In [ ]:
!{sys.executable} -m ladas2tei.main corpus/faux_visage_ONB_10790D2D/alto/<page_1> corpus/faux_visage_ONB_10790D2D/alto/<page_2> -o extrait.xml --title "Extrait"

## 3. Vérifier le résultat

**Combien de lignes, quelle taille ?**

In [ ]:
!wc -l TEI/faux_visage_ONB_10790D2D.xml
!ls -lh TEI

**Le début du fichier** (en-tête TEI et métadonnées) :

In [ ]:
!head -n 40 TEI/faux_visage_ONB_10790D2D.xml

**Quels éléments XML ont été produits, et combien de chacun ?** On enchaîne quatre commandes avec des tubes (`|`) :

In [ ]:
!grep -o "<[a-zA-Z]*" TEI/faux_visage_ONB_10790D2D.xml | sort | uniq -c | sort -rn | head

**Le fichier est-il du XML bien formé ?** `xmllint` est installé sur macOS ; sous Linux/WSL : `sudo apt install libxml2-utils`. Aucun message d'erreur = tout va bien.

In [ ]:
!xmllint --noout TEI/faux_visage_ONB_10790D2D.xml && echo "XML bien formé"

## 4. Les modes particuliers

Selon le type de texte, deux options changent la structure produite :

In [ ]:
!{sys.executable} -m ladas2tei.main corpus/faux_visage_ONB_10790D2D/alto -o article.xml --title "Article" --article
!{sys.executable} -m ladas2tei.main corpus/faux_visage_ONB_10790D2D/alto -o theatre.xml --title "Pièce" --theatre

Comparez les deux sorties avec la commande de comptage de la partie 3 :

In [ ]:
!grep -o "<[a-zA-Z]*" article.xml | sort | uniq -c | sort -rn | head
!echo "-----"
!grep -o "<[a-zA-Z]*" theatre.xml | sort | uniq -c | sort -rn | head

## 5. Les pièges classiques

| Symptôme | Cause probable |
|---|---|
| `No module named ladas2tei` | le carnet n'est pas dans le dossier `LADAS2TEI` (refaire `%cd`) |
| `No module named lxml` | le carnet n'utilise pas l'environnement virtuel (vérifier `sys.executable`) |
| erreur sur les métadonnées | nom du dossier différent de celui indiqué dans `metadata.csv` |
| aucun fichier produit | dossier `alto/` vide ou mal nommé |
| chemin introuvable | espace dans un nom de dossier : mettre des guillemets |

## Exercice

1. Lancer la conversion du dossier parent avec les métadonnées.
2. Retrouver dans le TEI le titre et l'auteur donnés dans `metadata.csv` (indice : `grep -n`).
3. Compter les éléments produits, et dire quel type d'élément est le plus fréquent.
4. Bonus : écrire un script `convertir.sh` qui active l'environnement, lance la conversion et affiche le décompte des éléments.

In [ ]:
# 1. Conversion

In [ ]:
# 2. Titre et auteur dans le TEI

In [ ]:
# 3. Décompte des éléments